In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.banking_gold
""")

print("✅ Gold schema created")

In [0]:
spark.sql("SHOW SCHEMAS IN workspace").show()

In [0]:
spark.sql("SHOW TABLES IN default").show()

In [0]:
customer_accounts_gold = spark.sql("""
SELECT
    c.customer_id,
    c.customer_name,
    c.email,
    c.age,
    c.city AS customer_city,
    c.state AS customer_state,

    a.account_id,
    a.account_type,
    a.opening_date,
    a.initial_balance,

    b.branch_id,
    b.branch_name,
    b.city AS branch_city,
    b.state AS branch_state

FROM default.customers_silver c

JOIN default.accounts_silver a
    ON c.customer_id = a.customer_id

LEFT JOIN default.branches_silver b
    ON a.branch_id = b.branch_id
""")

display(customer_accounts_gold)

In [0]:
customer_accounts_gold.write \
    .mode("overwrite") \
    .saveAsTable("workspace.banking_gold.customer_accounts")

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.banking_gold
""")

print("✅ Gold schema created")

In [0]:
spark.sql("SHOW SCHEMAS IN workspace").show()

In [0]:
spark.sql("""
SHOW TABLES IN banking_silver
""").show()

In [0]:
spark.sql("""
SELECT COUNT(*) AS total_records
FROM banking_gold.customer_accounts
""").show()

In [0]:
display(
    spark.table("banking_gold.customer_accounts").limit(10)
)

In [0]:
accounts = spark.table("default.accounts_silver")
customers = spark.table("default.customers_silver")
branches = spark.table("default.branches_silver")
merchants = spark.table("default.merchants_silver")
transactions = spark.table("default.transactions_silver")

print("Accounts:", accounts.count())
print("Customers:", customers.count())
print("Branches:", branches.count())
print("Merchants:", merchants.count())
print("Transactions:", transactions.count())

In [0]:
from pyspark.sql.functions import col

transactions = spark.table("default.transactions_silver")
accounts = spark.table("default.accounts_silver")
customers = spark.table("default.customers_silver")
branches = spark.table("default.branches_silver")
merchants = spark.table("default.merchants_silver")

transaction_details_gold = (
    transactions
    .join(accounts, "account_id", "left")
    .join(customers, "customer_id", "left")
    .join(branches, "branch_id", "left")
    .join(merchants, "merchant_id", "left")
)

print("Gold transaction records:", transaction_details_gold.count())

display(transaction_details_gold.limit(10))

In [0]:
from pyspark.sql.functions import col

# Load Silver tables
transactions = spark.table("default.transactions_silver")
accounts = spark.table("default.accounts_silver")
customers = spark.table("default.customers_silver")
branches = spark.table("default.branches_silver")
merchants = spark.table("default.merchants_silver")

# Rename overlapping columns
customers_clean = customers.select(
    "customer_id",
    "customer_name",
    "email",
    "age",
    col("city").alias("customer_city"),
    col("state").alias("customer_state"),
    "registration_date"
)

branches_clean = branches.select(
    "branch_id",
    "branch_name",
    col("city").alias("branch_city"),
    col("state").alias("branch_state")
)

merchants_clean = merchants.select(
    "merchant_id",
    "merchant_name",
    "merchant_category"
)

accounts_clean = accounts.select(
    "account_id",
    "customer_id",
    "branch_id",
    "account_type",
    "opening_date",
    "initial_balance"
)

# Build Gold transaction table
transaction_details_gold = (
    transactions
    .join(accounts_clean, "account_id", "left")
    .join(customers_clean, "customer_id", "left")
    .join(branches_clean, "branch_id", "left")
    .join(merchants_clean, "merchant_id", "left")
)

print("Gold transaction records:", transaction_details_gold.count())

display(transaction_details_gold.limit(10))

In [0]:
transaction_details_gold.printSchema()

In [0]:
transaction_details_gold.write \
    .mode("overwrite") \
    .saveAsTable("banking_gold.transaction_details")

In [0]:
spark.sql("""
SHOW TABLES IN banking_gold
""").show(100, False)

In [0]:
from pyspark.sql.functions import (
    col, count, sum, avg, when
)

# Load Gold transaction details
transaction_details = spark.table(
    "banking_gold.transaction_details"
)

# Customer-level KPIs
customer_kpis = (
    transaction_details
    .groupBy(
        "customer_id",
        "customer_name"
    )
    .agg(
        count("transaction_id").alias("total_transactions"),

        sum("amount").alias("total_transaction_amount"),

        avg("amount").alias("average_transaction_amount"),

        sum(
            when(col("status") == "SUCCESS", 1).otherwise(0)
        ).alias("successful_transactions"),

        sum(
            when(col("status") == "FAILED", 1).otherwise(0)
        ).alias("failed_transactions"),

        sum(
            when(col("status") == "PENDING", 1).otherwise(0)
        ).alias("pending_transactions"),

        sum(
            when(col("status") == "REVERSED", 1).otherwise(0)
        ).alias("reversed_transactions")
    )
)

display(customer_kpis.limit(10))

In [0]:
print("Customer KPI records:", customer_kpis.count())

In [0]:
customer_kpis.write \
    .mode("overwrite") \
    .saveAsTable("banking_gold.customer_kpis")

In [0]:
spark.sql("""
SHOW TABLES IN banking_gold
""").show(100, False)

In [0]:
branch_kpis = (
    transaction_details
    .groupBy(
        "branch_id",
        "branch_name",
        "branch_city",
        "branch_state"
    )
    .agg(
        count("transaction_id").alias("total_transactions"),

        sum("amount").alias("total_transaction_amount"),

        avg("amount").alias("average_transaction_amount"),

        sum(
            when(col("status") == "SUCCESS", 1).otherwise(0)
        ).alias("successful_transactions"),

        sum(
            when(col("status") == "FAILED", 1).otherwise(0)
        ).alias("failed_transactions")
    )
)

display(branch_kpis.limit(10))

In [0]:
print("Branch KPI records:", branch_kpis.count())

In [0]:
branch_kpis.write \
    .mode("overwrite") \
    .saveAsTable("banking_gold.branch_kpis")

In [0]:
from pyspark.sql.functions import col, count, sum, avg, when

# Load Gold transaction details
transaction_details = spark.table(
    "banking_gold.transaction_details"
)

# Merchant-level KPIs
merchant_kpis = (
    transaction_details
    .groupBy(
        "merchant_id",
        "merchant_name",
        "merchant_category"
    )
    .agg(
        count("transaction_id").alias("total_transactions"),

        sum("amount").alias("total_transaction_amount"),

        avg("amount").alias("average_transaction_amount"),

        sum(
            when(col("status") == "SUCCESS", 1).otherwise(0)
        ).alias("successful_transactions"),

        sum(
            when(col("status") == "FAILED", 1).otherwise(0)
        ).alias("failed_transactions"),

        sum(
            when(col("status") == "PENDING", 1).otherwise(0)
        ).alias("pending_transactions"),

        sum(
            when(col("status") == "REVERSED", 1).otherwise(0)
        ).alias("reversed_transactions")
    )
)

display(merchant_kpis.limit(10))

In [0]:
print("Merchant KPI records:", merchant_kpis.count())

In [0]:
merchant_kpis.write \
    .mode("overwrite") \
    .saveAsTable("banking_gold.merchant_kpis")

In [0]:
spark.sql("""
SHOW TABLES IN banking_gold
""").show(100, False)

In [0]:
spark.sql("""
SHOW TABLES IN banking_gold
""").show(100, False)

In [0]:
from pyspark.sql.functions import col, count, sum, avg, when

transactions = spark.table("default.transactions_silver")

overall_kpis = transactions.agg(
    count("transaction_id").alias("total_transactions"),
    sum("amount").alias("total_transaction_amount"),
    avg("amount").alias("average_transaction_amount"),

    sum(
        when(col("status") == "SUCCESS", 1).otherwise(0)
    ).alias("successful_transactions"),

    sum(
        when(col("status") == "FAILED", 1).otherwise(0)
    ).alias("failed_transactions"),

    sum(
        when(col("status") == "PENDING", 1).otherwise(0)
    ).alias("pending_transactions"),

    sum(
        when(col("status") == "REVERSED", 1).otherwise(0)
    ).alias("reversed_transactions")
)

display(overall_kpis)

In [0]:
success_rate = (
    transactions
    .agg(
        (
            sum(
                when(col("status") == "SUCCESS", 1).otherwise(0)
            ) / count("transaction_id") * 100
        ).alias("success_rate")
    )
)

display(success_rate)

In [0]:
transaction_type_kpis = (
    transactions
    .groupBy("transaction_type")
    .agg(
        count("transaction_id").alias("transaction_count"),
        sum("amount").alias("total_amount"),
        avg("amount").alias("average_amount")
    )
    .orderBy(col("transaction_count").desc())
)

display(transaction_type_kpis)

In [0]:
merchant_category_kpis = (
    transaction_details
    .groupBy("merchant_category")
    .agg(
        count("transaction_id").alias("transaction_count"),
        sum("amount").alias("total_amount"),
        avg("amount").alias("average_amount")
    )
    .orderBy(col("total_amount").desc())
)

display(merchant_category_kpis)

In [0]:
branch_performance = (
    spark.table("banking_gold.branch_kpis")
    .orderBy(
        col("total_transaction_amount").desc()
    )
)

display(branch_performance.limit(10))

In [0]:
transaction_details.filter(
    col("merchant_category").isNull()
).select(
    "transaction_id",
    "merchant_id",
    "amount",
    "status"
).show(20, False)

In [0]:
from pyspark.sql.functions import col, when

transaction_details_gold_clean = (
    transaction_details
    .withColumn(
        "merchant_category",
        when(
            col("merchant_category").isNull(),
            "UNKNOWN"
        ).otherwise(col("merchant_category"))
    )
)

display(
    transaction_details_gold_clean
    .filter(col("merchant_category") == "UNKNOWN")
    .limit(20)
)

In [0]:
transaction_details_gold_clean.groupBy(
    "merchant_category"
).count().orderBy(
    col("count").desc()
).show()

In [0]:
transaction_details_gold_clean.write \
    .mode("overwrite") \
    .saveAsTable("banking_gold.transaction_details")

In [0]:
gold_check = spark.table(
    "banking_gold.transaction_details"
)

print("Total Gold records:", gold_check.count())

print(
    "Duplicate transaction IDs:",
    gold_check.groupBy("transaction_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(
    "Null transaction IDs:",
    gold_check.filter(
        col("transaction_id").isNull()
    ).count()
)

print(
    "Null account IDs:",
    gold_check.filter(
        col("account_id").isNull()
    ).count()
)

print(
    "Null merchant categories:",
    gold_check.filter(
        col("merchant_category").isNull()
    ).count()
)

In [0]:
spark.sql("SHOW TABLES IN banking_gold").show(truncate=False)

In [0]:
from pyspark.sql.functions import col

print("========== GOLD TABLE COUNTS ==========")

gold_tables = [
    "customer_accounts",
    "customer_kpis",
    "branch_kpis",
    "merchant_kpis",
    "transaction_details"
]

for table in gold_tables:
    df = spark.table(f"banking_gold.{table}")
    print(f"{table:25} : {df.count():,}")

In [0]:
transactions_gold = spark.table("banking_gold.transaction_details")

print("Total records:", transactions_gold.count())

transactions_gold.show(5)

In [0]:
from pyspark.sql.functions import col

print("========== TRANSACTION GOLD QUALITY ==========")

print("Total records:", transactions_gold.count())

print(
    "Duplicate transaction IDs:",
    transactions_gold
        .groupBy("transaction_id")
        .count()
        .filter(col("count") > 1)
        .count()
)

print(
    "Null transaction IDs:",
    transactions_gold
        .filter(col("transaction_id").isNull())
        .count()
)

print(
    "Null account IDs:",
    transactions_gold
        .filter(col("account_id").isNull())
        .count()
)

print(
    "Null merchant categories:",
    transactions_gold
        .filter(col("merchant_category").isNull())
        .count()
)

In [0]:
from pyspark.sql.functions import count, countDistinct, sum, avg, round

transactions_gold.select(
    count("*").alias("total_transactions"),
    countDistinct("transaction_id").alias("unique_transactions"),
    round(sum("amount"), 2).alias("total_transaction_value"),
    round(avg("amount"), 2).alias("average_transaction_value")
).show()

In [0]:
from pyspark.sql.functions import col, count, sum, when, round

transactions_gold.groupBy("status").count().show()

success_rate = transactions_gold.select(
    round(
        sum(
            when(col("status") == "SUCCESS", 1).otherwise(0)
        ) / count("*") * 100,
        2
    ).alias("success_rate")
)

success_rate.show()

In [0]:
customer_kpis = spark.table("banking_gold.customer_kpis")

customer_kpis.printSchema()
customer_kpis.show(10, truncate=False)

In [0]:
from pyspark.sql.functions import (
    count,
    sum,
    avg,
    max,
    min,
    round
)

customer_kpis.select(
    count("*").alias("total_customers"),
    round(sum("total_transactions"), 0).alias("total_transactions"),
    round(sum("total_transaction_amount"), 2).alias("total_transaction_amount"),
    round(avg("total_transactions"), 2).alias("avg_transactions_per_customer"),
    round(avg("total_transaction_amount"), 2).alias("avg_amount_per_customer")
).show()

In [0]:
customer_kpis.orderBy(
    col("total_transaction_amount").desc()
).select(
    "customer_id",
    "customer_name",
    "total_transactions",
    "total_transaction_amount",
    "average_transaction_amount",
    "successful_transactions"
).show(10, truncate=False)

In [0]:
customer_kpis.orderBy(
    col("total_transactions").desc()
).select(
    "customer_id",
    "customer_name",
    "total_transactions",
    "total_transaction_amount",
    "successful_transactions"
).show(10, truncate=False)

In [0]:
from pyspark.sql.functions import count, sum, avg, round

customer_kpis.select(
    count("*").alias("total_customers"),
    sum("total_transactions").alias("total_transactions"),
    round(sum("total_transaction_amount"), 2).alias("total_transaction_value"),
    round(avg("total_transactions"), 2).alias("avg_transactions_per_customer"),
    round(avg("total_transaction_amount"), 2).alias("avg_value_per_customer")
).show()

In [0]:
branch_kpis = spark.table("banking_gold.branch_kpis")

branch_kpis.printSchema()
branch_kpis.show(10, truncate=False)

In [0]:
from pyspark.sql.functions import count, sum, avg, round

branch_kpis.select(
    count("*").alias("total_branches"),
    sum("total_transactions").alias("total_transactions"),
    round(sum("total_transaction_amount"), 2).alias("total_transaction_value"),
    round(avg("total_transactions"), 2).alias("avg_transactions_per_branch"),
    round(avg("total_transaction_amount"), 2).alias("avg_value_per_branch")
).show()

In [0]:
branch_kpis.orderBy(
    col("total_transaction_amount").desc()
).select(
    "branch_id",
    "branch_name",
    "branch_city",
    "branch_state",
    "total_transactions",
    "total_transaction_amount",
    "average_transaction_amount",
    "successful_transactions",
    "failed_transactions"
).show(10, truncate=False)

In [0]:
merchant_kpis = spark.table("banking_gold.merchant_kpis")

merchant_kpis.printSchema()
merchant_kpis.show(10, truncate=False)

In [0]:
from pyspark.sql.functions import count, sum, avg, round

merchant_kpis.select(
    count("*").alias("total_merchants"),
    sum("total_transactions").alias("total_transactions"),
    round(sum("total_transaction_amount"), 2).alias("total_transaction_value"),
    round(avg("total_transactions"), 2).alias("avg_transactions_per_merchant"),
    round(avg("total_transaction_amount"), 2).alias("avg_value_per_merchant")
).show()


In [0]:
merchant_kpis.filter(
    col("merchant_id").isNull()
).show(truncate=False)

In [0]:
merchant_kpis.filter(
    col("merchant_category") == "UNKNOWN"
).count()

In [0]:
merchant_kpis.select(
    "merchant_id"
).distinct().count()

In [0]:
from pyspark.sql.functions import col

merchant_kpis_clean = merchant_kpis.filter(
    col("merchant_id") != "UNKNOWN"
)

merchant_kpis_clean.write \
    .mode("overwrite") \
    .saveAsTable("banking_gold.merchant_kpis")

In [0]:
merchant_kpis = spark.table("banking_gold.merchant_kpis")

print("Silver merchants:", spark.table("banking_silver.merchants").count())
print("Gold merchants:", merchant_kpis.select("merchant_id").distinct().count())

merchant_kpis.filter(
    col("merchant_id") == "UNKNOWN"
).show()

In [0]:
spark.sql("SHOW TABLES IN banking_gold").show(truncate=False)